# **Day 34: Reading and Writing Text Files**

Till now, all our data disappeared when the program ended. **Files** let us save data permanently.

We open a file with `open(path, mode)`:

| Mode | Meaning |
|---|---|
| `"r"` | Read (default) |
| `"w"` | Write (**deletes** old content first!) |
| `"a"` | Append (add at the end) |
| `"x"` | Create a new file (error if it already exists) |
| `"b"` | Binary mode, used with others like `"rb"` |

We always use `with open(...)`. It **closes the file automatically**, even if an error happens.

## **Writing a File**

In [1]:
with open("poem.txt", "w", encoding="utf-8") as f:
    f.write("Kolkata, city of joy\n")
    f.write("trams and yellow taxis\n")
    f.write("phuchka at every corner\n")
print("File written")

File written


## **Reading a File**

In [2]:
with open("poem.txt", "r", encoding="utf-8") as f:
    content = f.read()        # Read everything at once
print(content)

Kolkata, city of joy
trams and yellow taxis
phuchka at every corner



In [3]:
# Read one line at a time
with open("poem.txt", encoding="utf-8") as f:
    print(f.readline(), end="")
    print(f.readline(), end="")

Kolkata, city of joy
trams and yellow taxis


In [4]:
# Read all lines into a list
with open("poem.txt", encoding="utf-8") as f:
    lines = f.readlines()
print(lines)

['Kolkata, city of joy\n', 'trams and yellow taxis\n', 'phuchka at every corner\n']


In [5]:
# The best way for big files: loop over the file (reads one line at a time)
with open("poem.txt", encoding="utf-8") as f:
    for number, line in enumerate(f, 1):
        print(number, line.strip())

1 Kolkata, city of joy
2 trams and yellow taxis
3 phuchka at every corner


## **Appending to a File**

In [6]:
with open("poem.txt", "a", encoding="utf-8") as f:
    f.write("and the Howrah Bridge at night\n")

with open("poem.txt", encoding="utf-8") as f:
    print(f.read())

Kolkata, city of joy
trams and yellow taxis
phuchka at every corner
and the Howrah Bridge at night



## **Bengali Text Needs encoding="utf-8"**
On Windows, if we forget `encoding="utf-8"`, Bengali text can show as strange symbols or give an error.

In [7]:
with open("bangla.txt", "w", encoding="utf-8") as f:
    f.write("আমার সোনার বাংলা\n")

with open("bangla.txt", encoding="utf-8") as f:
    print(f.read())

আমার সোনার বাংলা



## **Example from Remote Sensing: Reading a Metadata File**
When we download a Landsat image, it comes with a text file called the **metadata file** (its name ends with `_MTL.txt`). It has information in the form `KEY = VALUE`.

Let us create a small version of such a file and then read it into a dictionary.

In [8]:
metadata_text = """GROUP = PRODUCT_METADATA
  SPACECRAFT_ID = "LANDSAT_8"
  SENSOR_ID = "OLI_TIRS"
  WRS_PATH = 138
  WRS_ROW = 44
  DATE_ACQUIRED = 2023-11-10
  SCENE_CENTER_TIME = "04:36:12.5Z"
  CLOUD_COVER = 2.35
  SUN_ELEVATION = 45.12
END_GROUP = PRODUCT_METADATA
"""
with open("LC08_sample_MTL.txt", "w") as f:
    f.write(metadata_text)

In [9]:
meta = {}
with open("LC08_sample_MTL.txt") as f:
    for line in f:
        line = line.strip()
        if "=" not in line or line.startswith(("GROUP", "END_GROUP")):
            continue
        key, value = line.split("=", 1)
        meta[key.strip()] = value.strip().strip('"')

for k, v in meta.items():
    print(f"{k:<18}: {v}")

print("\nCloud cover is", float(meta["CLOUD_COVER"]), "%")

SPACECRAFT_ID     : LANDSAT_8
SENSOR_ID         : OLI_TIRS
WRS_PATH          : 138
WRS_ROW           : 44
DATE_ACQUIRED     : 2023-11-10
SCENE_CENTER_TIME : 04:36:12.5Z
CLOUD_COVER       : 2.35
SUN_ELEVATION     : 45.12

Cloud cover is 2.35 %


Let us understand the code above:
- `line.split("=", 1)`: Split only at the **first** `=` sign, into a key and a value.
- `.strip('"')`: Removes the quote marks around text values.
- We skip the `GROUP` lines because they do not hold data.
- All values come in as text, so we use `float()` when we need a number.

## **Writing Many Lines and print() to a File**

In [10]:
stations = ["Howrah", "Bally", "Belur", "Liluah", "Serampore"]

with open("stations.txt", "w", encoding="utf-8") as f:
    f.writelines(s + "\n" for s in stations)
    print("Total:", len(stations), file=f)       # print() can also write into a file

print(open("stations.txt", encoding="utf-8").read())

Howrah
Bally
Belur
Liluah
Serampore
Total: 5



## **Reading a File That Does Not Exist**

In [11]:
# open("does_not_exist.txt")     # This will raise a FileNotFoundError
from pathlib import Path
print(Path("does_not_exist.txt").exists())

False


## **Practice Problem: Word Counter**
Count the lines, words and characters of a file.

In [12]:
def word_count(path):
    lines = words = chars = 0
    with open(path, encoding="utf-8") as f:
        for line in f:
            lines += 1
            words += len(line.split())
            chars += len(line)
    return lines, words, chars

print(word_count("poem.txt"))

(4, 18, 99)


In [13]:
# Quick question
# What will happen to the old content of the file?
# with open("poem.txt", "w") as f:
#     f.write("new")